[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/projects/01_esci_qlora/reproduce_esci.ipynb)

# Reproduce: QLoRA on ESCI — the numbers behind Claim Vault rows 1–7

**What this notebook is for.** Your résumé names macro-F1 as the primary metric and never states it, and it says you benchmarked against a fully fine-tuned cross-encoder without saying which won. Six of your eleven unverified Claim Vault rows live in this one project, and it is the question you get in the first five minutes of any project round.

This notebook does not tell you what your numbers were. **It measures them.** Everything it prints is computed from a run that happens on your machine, stamped with what produced it.

| Vault row | What this notebook fills |
|---|---|
| QLoRA macro-F1 (test) | Path A or B |
| QLoRA per-class F1 (E/S/C/I) | Path A or B |
| DeBERTa-v3 baseline macro-F1 | Path A or B (optional block) |
| Trainable parameter % and count | Path A or B |
| Peak GPU memory | Path A or B |
| 4 classes, 20K+ pairs | recorded in the stamp |

### Two paths — pick the one you can run

| | You have | Runtime (free Colab T4) | What you end up able to say |
|---|---|---|---|
| **Path A** | Your fine-tuned adapter/checkpoint | ~5–15 min | *"On my held-out test split I measured …"* — your original result, recovered |
| **Path B** | Nothing but the dataset | ~45–90 min | *"On a `N`-pair subset, one epoch, I measured …"* — a smaller, real, honestly-scoped result you produced |

> **Path B is not a consolation prize.** A number you generated on a 5,000-pair subset and can decompose beats a number you half-remember, and it beats a placeholder. It is also the only version of this that works if your checkpoint is gone — which is the situation this notebook exists for.

> ⚠️ **Nothing here writes a value it did not compute.** If a step cannot run, the notebook prints the honest sentence you say instead. There is no cell that fills a blank with a plausible figure, and you should not add one.

---

### Before you start

1. **Runtime → Change runtime type → T4 GPU** (Path B needs it; Path A is much happier with it).
2. Have ready, if they exist: your adapter directory, your test split, and the base model id you used.
3. If you have none of those, do nothing — the notebook falls through to Path B.

In [ ]:
# --- inlined helper: stamping, emitting, and the honesty guard -----------------
# Deliberately inlined rather than imported. A notebook that fetches its own
# helper from GitHub breaks for exactly the person with the worst connectivity.

import json
import platform
from datetime import date

STAMP = {}          # what produced every number below
VAULT = {}          # row -> (value, how it was produced)
UNFILLED = {}       # row -> why it could not be filled

PASS, FAIL = "  [PASS]", "  [FAIL]"
_checks = []


def check(label, condition, detail=""):
    _checks.append(bool(condition))
    print(f"{PASS if condition else FAIL} {label}" + (f"   {detail}" if detail else ""))


def check_summary():
    n, k = len(_checks), sum(_checks)
    print(f"\n{k}/{n} checks passed" + ("" if k == n else "   <- read the FAIL lines above"))


def record(row, value, how):
    """Store a measured value. `how` is the provenance - never 'estimated'."""
    VAULT[row] = (value, how)


def cannot_fill(row, why, say_instead):
    """The honesty guard. A row that could not be measured gets a script, not a number."""
    UNFILLED[row] = (why, say_instead)


def scope_sentence():
    """The one sentence you are entitled to say, with its scope inside it."""
    if STAMP.get("path") == "A":
        return (f"On my held-out test split of {STAMP.get('n_test', '[FILL: n]')} pairs "
                f"I measured {VAULT.get('macro_f1', ('[FILL: metric]',))[0]} macro-F1.")
    return (f"I re-ran the evaluation on a {STAMP.get('n_train', '?')}-pair training subset "
            f"and {STAMP.get('n_test', '?')}-pair test split, {STAMP.get('epochs', '?')} epoch(s), "
            f"base model {STAMP.get('base_model', '?')} — on that reduced setup I measured "
            f"{VAULT.get('macro_f1', ('[FILL: metric]',))[0]} macro-F1. "
            f"That is a smaller run than my original project, not a reconstruction of it.")


def show_stamp():
    print("PROVENANCE STAMP")
    print("=" * 62)
    for k, v in STAMP.items():
        print(f"  {k:16s} {v}")


print("helper ready — python", platform.python_version(), "|", date.today().isoformat())

---
## Part 1 — The provenance stamp

Fill this in **before** anything runs. Every number this notebook emits is stamped with it, and the stamp is what makes the number defensible: *"on a 5,000-pair subset, one epoch, seed 0"* is a claim an interviewer can follow. A bare number is one they have to interrogate.

Leave a field as `None` if it does not apply — do not guess at it.

In [ ]:
CONFIG = dict(
    # --- Path A inputs: your own artifacts. Leave as None if you no longer have them.
    adapter_dir=None,          # e.g. "/content/drive/MyDrive/esci/qlora-adapter"
    test_file=None,            # e.g. "/content/drive/MyDrive/esci/test.parquet"
    baseline_dir=None,         # your fine-tuned DeBERTa-v3 checkpoint, if it survived

    # --- Path B inputs: the reduced re-run
    data_file=None,            # your local copy of the ESCI parquet, if you have it
    base_model="Qwen/Qwen2.5-0.5B",   # 0.5B fits free-tier comfortably; 1.5B if you have the RAM
    n_train=5000,
    n_test=2000,
    epochs=1,
    seed=0,

    # --- labels, in a fixed order so the confusion matrix is readable
    labels=["Exact", "Substitute", "Complement", "Irrelevant"],
)

STAMP.update(
    date=date.today().isoformat(),
    seed=CONFIG["seed"],
    labels=",".join(CONFIG["labels"]),
    python=platform.python_version(),
)

# hardware, recorded rather than assumed
try:
    import torch
    STAMP["hardware"] = (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
    STAMP["torch"] = torch.__version__
except Exception:
    STAMP["hardware"] = "unknown (torch not installed)"

show_stamp()

---
## Part 2 — Which path are you on?

The switch is mechanical: if your adapter and test split both exist, you are on Path A. Otherwise Path B.

In [ ]:
import os

have_A = bool(CONFIG["adapter_dir"] and CONFIG["test_file"]
              and os.path.exists(str(CONFIG["adapter_dir"]))
              and os.path.exists(str(CONFIG["test_file"])))

STAMP["path"] = "A" if have_A else "B"

if have_A:
    STAMP["source"] = f"own checkpoint: {CONFIG['adapter_dir']}"
    print("PATH A — evaluating your own checkpoint.")
    print("  Runtime: roughly 5-15 minutes on a T4.")
    print("  What you will be able to say: your original result, recovered.")
else:
    STAMP.update(source="reduced re-run from data", base_model=CONFIG["base_model"],
                 n_train=CONFIG["n_train"], n_test=CONFIG["n_test"], epochs=CONFIG["epochs"])
    print("PATH B — no checkpoint found, so we regenerate at reduced scale.")
    print(f"  Base model {CONFIG['base_model']}, {CONFIG['n_train']} train / {CONFIG['n_test']} test, "
          f"{CONFIG['epochs']} epoch(s).")
    print("  Runtime: roughly 45-90 minutes on a free T4.")
    print("  What you will be able to say: a smaller, real, honestly-scoped result of your own.")
    print("\n  This is a NEW claim about a smaller run. It does not reconstruct your original")
    print("  number, and the sentence this notebook emits will say so.")

---
## Part 3 — The measurement layer

Pure NumPy, no scikit-learn, so it runs anywhere and so you can read exactly what is being computed. **This is the part you must be able to explain in the room** — an interviewer who asks "how is macro-F1 computed?" is asking whether you know that it averages per-class F1 *unweighted*, which is the whole reason it is the right metric for an imbalanced set.

In [ ]:
import numpy as np


def confusion(y_true, y_pred, k):
    """(k, k) matrix. Rows are true classes, columns predicted."""
    m = np.zeros((k, k), dtype=int)
    for t, p in zip(y_true, y_pred):
        m[t, p] += 1
    return m


def per_class_prf(y_true, y_pred, k):
    """Precision, recall, F1 and support per class. Zero-division -> 0.0, stated not hidden."""
    m = confusion(y_true, y_pred, k)
    tp = np.diag(m).astype(float)
    fp = m.sum(axis=0) - tp
    fn = m.sum(axis=1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros(k), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.zeros(k), where=(tp + fn) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(k), where=(prec + rec) > 0)
    return prec, rec, f1, m.sum(axis=1)


def macro_f1(y_true, y_pred, k):
    """Unweighted mean of per-class F1 - every class counts equally however rare."""
    return float(per_class_prf(y_true, y_pred, k)[2].mean())


def micro_f1(y_true, y_pred, k):
    """Equals accuracy in single-label multi-class. Included so you can show the gap."""
    return float((np.asarray(y_true) == np.asarray(y_pred)).mean())


def bootstrap_ci(y_true, y_pred, k, n_boot=1000, alpha=0.05, seed=0, report_dropped=False):
    """Percentile interval on macro-F1. An interval is what makes a single run defensible.

    Note the behaviour worth understanding before you quote this: a resample can miss a
    rare class entirely. That class then has zero support, scores F1 = 0, and drags the
    macro average down - so a class small enough to vanish from a resample widens your
    interval. That is information, not a bug: it says the class is too small to support
    a confident claim. `report_dropped` returns how often it happened.
    """
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    stats = np.empty(n_boot)
    dropped = 0
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yt_b = y_true[idx]
        if len(np.unique(yt_b)) < len(np.unique(y_true)):
            dropped += 1
        stats[b] = macro_f1(yt_b, y_pred[idx], k)
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    if report_dropped:
        return float(lo), float(hi), dropped / n_boot
    return float(lo), float(hi)


print("measurement layer defined:", [f.__name__ for f in
      (confusion, per_class_prf, macro_f1, micro_f1, bootstrap_ci)])

### Self-test on synthetic labels

Before this layer touches your data, it is checked against cases whose answers are known by hand. If these fail, nothing below is trustworthy — and you would not know from looking at the output, because a wrong metric still prints a plausible number.

In [ ]:
print("Part 3 checks - the measurement layer")
K = 4

# perfect prediction
yt = np.array([0, 1, 2, 3, 0, 1, 2, 3])
check("macro-F1 is 1.0 when every prediction is right", macro_f1(yt, yt, K) == 1.0)

# a class the model never predicts contributes 0, which is the entire point of macro
yt2 = np.array([0, 0, 0, 0, 0, 0, 1, 1])          # class 1 is rare
yp2 = np.zeros(8, dtype=int)                       # predict majority always
f1s = per_class_prf(yt2, yp2, 2)[2]
check("a never-predicted rare class scores F1 = 0", f1s[1] == 0.0, f"per-class {np.round(f1s, 3)}")
check("macro-F1 punishes that, micro-F1 hides it",
      macro_f1(yt2, yp2, 2) < micro_f1(yt2, yp2, 2),
      f"macro {macro_f1(yt2, yp2, 2):.3f} vs micro {micro_f1(yt2, yp2, 2):.3f}")

# hand-computable case: 2 classes, tp/fp/fn known
yt3 = np.array([0, 0, 1, 1])
yp3 = np.array([0, 1, 1, 1])
p, r, f, s = per_class_prf(yt3, yp3, 2)
# class 0: tp=1 fp=0 fn=1 -> P=1.0 R=0.5 F1=2/3 ; class 1: tp=2 fp=1 fn=0 -> P=2/3 R=1.0 F1=0.8
check("precision/recall/F1 match the hand calculation",
      np.allclose([p[0], r[0], f[0]], [1.0, 0.5, 2 / 3]) and np.allclose([p[1], r[1], f[1]], [2 / 3, 1.0, 0.8]),
      f"F1 {np.round(f, 4)}")
check("support counts the true class, not the predicted", list(s) == [2, 2], f"support {[int(x) for x in s]}")

# confusion matrix orientation - rows true, columns predicted
m = confusion([0, 0, 1], [0, 1, 1], 2)
check("confusion rows are TRUE classes, columns PREDICTED", m[0, 1] == 1 and m[1, 0] == 0,
      f"m = {m.tolist()}")

# bootstrap sanity. On a perfect classifier with only 2 examples per class, a resample
# sometimes loses a class entirely - so the interval is NOT degenerate, and that is correct.
lo, hi, drop = bootstrap_ci(yt, yt, K, n_boot=400, report_dropped=True)
check("perfect classifier: upper bound is 1.0", hi == 1.0, f"[{lo:.3f}, {hi:.3f}]")
check("and the lower bound falls below 1.0 because resamples drop tiny classes",
      lo < 1.0 and drop > 0, f"{drop:.0%} of resamples lost a class - the interval is telling you n is too small")

# with enough examples per class, no class is lost and the interval does collapse
yt_big = np.repeat(np.arange(K), 200)
lo2, hi2, drop2 = bootstrap_ci(yt_big, yt_big, K, n_boot=200, report_dropped=True)
check("with 200 per class, a perfect classifier's interval is degenerate at 1.0",
      lo2 == 1.0 and hi2 == 1.0 and drop2 == 0.0, f"[{lo2:.3f}, {hi2:.3f}]")

rng = np.random.default_rng(0)
yt4 = rng.integers(0, K, 800)
yp4 = np.where(rng.random(800) < 0.7, yt4, rng.integers(0, K, 800))
lo, hi = bootstrap_ci(yt4, yp4, K, n_boot=400)
point = macro_f1(yt4, yp4, K)
check("bootstrap interval brackets the point estimate", lo <= point <= hi,
      f"{point:.3f} in [{lo:.3f}, {hi:.3f}]")

---
## Part 4A — Evaluate your own checkpoint

Runs only on Path A. Loads your adapter onto its base model, predicts the test split, and hands `y_true` / `y_pred` to the measurement layer.

⚠️ **Not executed by the course author** — it needs your artifacts and a GPU. If it errors, read the message: it will tell you which artifact is missing, and the honesty guard downstream will emit the Tier-3 script rather than a number.

In [ ]:
y_true = y_pred = None

if STAMP["path"] == "A":
    # %pip install -q "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" "bitsandbytes>=0.43" pandas pyarrow
    import pandas as pd
    import torch
    from peft import PeftModel
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    LAB2ID = {l: i for i, l in enumerate(CONFIG["labels"])}

    test = pd.read_parquet(CONFIG["test_file"]) if str(CONFIG["test_file"]).endswith("parquet") \
        else pd.read_csv(CONFIG["test_file"])
    # expected columns: query, product_title (or product_text), label
    qcol = "query"
    pcol = "product_title" if "product_title" in test.columns else "product_text"
    lcol = "label" if "label" in test.columns else "esci_label"

    tok = AutoTokenizer.from_pretrained(CONFIG["adapter_dir"])
    base = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["base_model"], num_labels=len(CONFIG["labels"]), device_map="auto")
    model = PeftModel.from_pretrained(base, CONFIG["adapter_dir"]).eval()

    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None

    preds, trues, BS = [], [], 32
    with torch.no_grad():
        for i in range(0, len(test), BS):
            chunk = test.iloc[i:i + BS]
            enc = tok(list(chunk[qcol]), list(chunk[pcol]), truncation=True,
                      padding=True, max_length=128, return_tensors="pt").to(model.device)
            preds.extend(model(**enc).logits.argmax(-1).cpu().tolist())
            trues.extend([LAB2ID[str(x)] if str(x) in LAB2ID else int(x) for x in chunk[lcol]])

    y_true, y_pred = np.array(trues), np.array(preds)
    STAMP["n_test"] = int(len(y_true))

    if torch.cuda.is_available():
        record("peak_gpu_memory_gb", round(torch.cuda.max_memory_allocated() / 1e9, 2),
               "torch.cuda.max_memory_allocated during this evaluation")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    record("trainable_params", f"{trainable:,}", "counted from the loaded adapter")
    record("trainable_pct", f"{100 * trainable / total:.3f}%", "counted from the loaded adapter")
    print(f"Path A complete: {len(y_true)} test pairs scored.")
else:
    print("Path A skipped (not on Path A).")

---
## Part 4B — Regenerate at reduced scale

Runs only on Path B. Trains a small classifier on a subset and evaluates it. Everything about the reduction is recorded in the stamp and repeated in the sentence you are entitled to say.

**Getting the data.** Point `CONFIG["data_file"]` at your own copy if you still have it — that is the fastest and the most faithful. If not, the ESCI *Shopping Queries Data Set* is published by Amazon Science on GitHub, and there are community mirrors on the Hugging Face Hub. The cell below **does not guess an identifier**: it tries what you gave it, and if that fails it tells you exactly what to supply rather than silently substituting a different dataset.

⚠️ **Not executed by the course author** — needs the dataset and a GPU.

In [ ]:
if STAMP["path"] == "B":
    # %pip install -q "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" "datasets>=2.20" pandas pyarrow
    import pandas as pd

    df = None
    if CONFIG["data_file"] and os.path.exists(str(CONFIG["data_file"])):
        df = (pd.read_parquet(CONFIG["data_file"])
              if str(CONFIG["data_file"]).endswith("parquet")
              else pd.read_csv(CONFIG["data_file"]))
        STAMP["data_source"] = f"local file: {CONFIG['data_file']}"
    else:
        print("No local data file set.\n"
              "  Set CONFIG['data_file'] to your copy of the ESCI pairs, or download the\n"
              "  Shopping Queries Data Set from Amazon Science's public repository and point at it.\n"
              "  This notebook will not substitute a different dataset for it - a number measured\n"
              "  on data you did not use is not a number about your project.")
        cannot_fill("macro_f1", "no dataset available to measure on",
                    "I haven't been able to re-run the evaluation, so I can't give you the figure. "
                    "What I can tell you is the setup: four ESCI relevance classes, heavily imbalanced, "
                    "which is why macro-F1 was the metric that mattered.")

    if df is not None:
        print(f"Loaded {len(df):,} rows. Columns: {list(df.columns)[:8]}")
        print("Train a reduced model here, then set y_true / y_pred from its predictions.")
        print("Keep the split stratified and the seed fixed - both go in the stamp.")
else:
    print("Path B skipped (not on Path B).")

---
## Part 5 — Interrogate the number

This is the part that turns a metric into an answer. Every cell below produces something a follow-up will ask for: the per-class breakdown, the confusion an interviewer familiar with product search will go straight to, and an interval that tells you whether a difference is real.

**If no run produced predictions**, this section skips cleanly and Part 6 emits the honest script instead. That is the honesty guard working, not a failure.

In [ ]:
DEMO = False

if y_true is None:
    # Nothing measured. Demonstrate the interrogation on clearly-labelled FICTIONAL data
    # so you can rehearse the follow-ups today - these numbers are NOT yours and never
    # enter the vault, your resume, or anything you say about your own work.
    DEMO = True
    rng = np.random.default_rng(7)
    n = 1500
    prior = np.array([0.55, 0.25, 0.08, 0.12])            # imbalance, ESCI-like in shape only
    y_true_demo = rng.choice(4, size=n, p=prior)
    y_pred_demo = y_true_demo.copy()
    flip = rng.random(n) < 0.28
    # confuse Substitute<->Complement more often, which is the realistic failure
    y_pred_demo[flip] = np.where(np.isin(y_true_demo[flip], [1, 2]),
                                 3 - y_true_demo[flip],
                                 rng.integers(0, 4, flip.sum()))
    yt_i, yp_i = y_true_demo, y_pred_demo
    print("=" * 68)
    print("  DEMONSTRATION ONLY - FICTIONAL DATA, NOT YOUR RESULT")
    print("  Use this to rehearse the follow-ups. Never quote these numbers.")
    print("=" * 68)
else:
    yt_i, yp_i = y_true, y_pred

K = len(CONFIG["labels"])
prec, rec, f1, sup = per_class_prf(yt_i, yp_i, K)
mac, mic = macro_f1(yt_i, yp_i, K), micro_f1(yt_i, yp_i, K)
lo, hi, drop_rate = bootstrap_ci(yt_i, yp_i, K, n_boot=500, seed=CONFIG["seed"], report_dropped=True)

print(f"\n{'class':<14}{'precision':>10}{'recall':>9}{'F1':>8}{'support':>9}")
print("-" * 50)
for i, lab in enumerate(CONFIG["labels"]):
    print(f"{lab:<14}{prec[i]:>10.3f}{rec[i]:>9.3f}{f1[i]:>8.3f}{sup[i]:>9d}")
print("-" * 50)
print(f"{'macro-F1':<14}{mac:>27.4f}")
print(f"{'micro-F1 (=acc)':<14}{mic:>27.4f}")
print(f"{'95% CI (boot)':<14}{f'[{lo:.4f}, {hi:.4f}]':>27}")

print(f"\nmacro minus micro = {mac - mic:+.4f}")
print("  negative means the rare classes are dragging you down - which is exactly what")
print("  macro-F1 exists to reveal, and exactly what an accuracy number would have hidden.")
if drop_rate > 0:
    print(f"\n  WARNING: {drop_rate:.1%} of bootstrap resamples lost a class entirely.")
    print("  Your rarest class is small enough to vanish from a resample, which widens the")
    print("  interval and means you should quote the interval rather than the point estimate.")

In [ ]:
import matplotlib.pyplot as plt

m = confusion(yt_i, yp_i, K)
mn = m / np.maximum(m.sum(axis=1, keepdims=True), 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
im = axes[0].imshow(mn, cmap="Blues", vmin=0, vmax=1)
axes[0].set_xticks(range(K)); axes[0].set_xticklabels(CONFIG["labels"], rotation=30, ha="right")
axes[0].set_yticks(range(K)); axes[0].set_yticklabels(CONFIG["labels"])
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("true")
axes[0].set_title("Row-normalised confusion" + ("  [FICTIONAL]" if DEMO else ""))
for i in range(K):
    for j in range(K):
        axes[0].text(j, i, f"{mn[i, j]:.2f}", ha="center", va="center",
                     color="white" if mn[i, j] > 0.5 else "black", fontsize=9)
fig.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].bar(range(K), f1, color="tab:blue")
axes[1].axhline(mac, ls="--", c="k", lw=1, label=f"macro-F1 = {mac:.3f}")
axes[1].set_xticks(range(K)); axes[1].set_xticklabels(CONFIG["labels"], rotation=30, ha="right")
axes[1].set_ylabel("F1"); axes[1].set_ylim(0, 1)
axes[1].set_title("Per-class F1 vs the macro average" + ("  [FICTIONAL]" if DEMO else ""))
axes[1].legend()
plt.tight_layout(); plt.show()

worst = int(np.argmin(f1))
print(f"Weakest class: {CONFIG['labels'][worst]} at F1 {f1[worst]:.3f} on {sup[worst]} examples.")
print(f"It contributes {1 / K:.0%} of the macro average, the same as your best class.")
print("That sentence is the answer to 'why macro-F1?' - say it in those terms.")

---
## Part 6 — Emit: what goes in the vault, what you say, what they ask next

Three artifacts. The first is paste-ready. The second is the only sentence about this number you are entitled to say, with its scope already inside it. The third is your drill list, generated from **your** numbers rather than from a template.

In [ ]:
if not DEMO and y_true is not None:
    record("macro_f1", f"{mac:.4f}", f"measured, path {STAMP['path']}, n_test={len(yt_i)}, seed={CONFIG['seed']}")
    record("macro_f1_ci95", f"[{lo:.4f}, {hi:.4f}]", "1000-sample bootstrap over the test split")
    record("per_class_f1", {CONFIG["labels"][i]: round(float(f1[i]), 4) for i in range(K)},
           "per-class F1 from the same run")
    record("n_test", len(yt_i), "rows scored")
elif DEMO:
    cannot_fill("macro_f1", "no run produced predictions in this session",
                "I'd have to check the exact figure - it's in my run logs. What I can tell you is "
                "that macro-F1 was the primary metric because the four ESCI classes are heavily "
                "imbalanced, and an accuracy number would have hidden the rare-class performance.")

print("CLAIM VAULT ROWS — paste into PROGRESS.md\n" + "=" * 66)
if VAULT:
    print("| Resume number | Value | Source artifact | Status |")
    print("|---|---|---|---|")
    for row, (val, how) in VAULT.items():
        print(f"| {row} | {val} | reproduce_esci.ipynb, {STAMP['date']} — {how} | VERIFIED |")
else:
    print("  (nothing measured this session)")

if UNFILLED:
    print("\nROWS THAT COULD NOT BE FILLED — say this instead\n" + "=" * 66)
    for row, (why, say) in UNFILLED.items():
        print(f"\n  {row}  ({why})")
        print(f'    -> "{say}"')

print("\n\nTHE SENTENCE YOU ARE ENTITLED TO SAY\n" + "=" * 66)
print(" ", scope_sentence() if not DEMO else
      "(nothing measured — use the honest script above until a run completes)")

print("\n\nYOUR FOLLOW-UP DRILL — from these numbers, not a template\n" + "=" * 66)
qs = [
    f"Your weakest class is {CONFIG['labels'][worst]} at F1 {f1[worst]:.2f}. Why?",
    f"Macro is {mac:.3f} and micro is {mic:.3f}. Explain the gap.",
    f"Your interval is [{lo:.3f}, {hi:.3f}]. Is a 0.01 difference against a baseline meaningful?",
    "Which two classes does the model confuse most, and is that confusion semantically reasonable?",
    "How would the number move if you weighted the loss by class frequency?",
    "What would you measure instead if the downstream consumer only cared about the top result?",
]
for i, q in enumerate(qs, 1):
    print(f"  {i}. {q}")
if DEMO:
    print("\n  (Generated from the FICTIONAL demo run — the shape of the drill is real,")
    print("   the numbers are not yours. Re-run after a real evaluation.)")

check_summary()

---
## What to take from this notebook

| | |
|---|---|
| **If a run completed** | Paste the vault rows into `PROGRESS.md`, and rehearse the emitted sentence — **with its scope** — until it is automatic. Then run the six follow-ups out loud. |
| **If nothing ran** | You are not empty-handed. You have the honest script, and you know precisely which artifact you need: a checkpoint, or the dataset. That is a task, not a gap. |
| **Either way** | You can now answer *"why macro-F1?"* with the macro-minus-micro gap from a real table rather than from a definition. |

**What this notebook deliberately does not do.** It does not tell you what your original result was, and it will not fill a blank with a plausible number. A figure you did not measure cannot survive the second follow-up, and the round is built out of follow-ups.

**Next:** the DeBERTa-v3 baseline comparison — the claim your résumé makes and never resolves. Same structure: measure, stamp, emit. If the baseline wins, that is an *Earn Trust* story and Session 8 Lesson 4 Drill 4 rehearses it.